# RFdiffusion3 Structure Design

![RFdiffusion3 Structure Design](https://proto-bio.github.io/proto-assets/images/tool/rfdiffusion3/hero.png)

This notebook demonstrates de novo protein structure design using RFdiffusion3. We generate a novel protein backbone from scratch by specifying a target length, then inspect the designed sequence and structure returned by the model.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("rfdiffusion3")
display_overview("rfdiffusion3")
display_docs_section("rfdiffusion3", "Background")

## Available tools

In [ ]:
display_available_tools("rfdiffusion3")

### `run_rfdiffusion3`

RFdiffusion3 is an all-atom diffusion model that co-designs protein backbone structure and amino acid sequence. Starting from a design specification — which can be as simple as a target length or as detailed as a set of motif constraints and hotspot residues — the model iteratively denoises random coordinates into a physically plausible protein. This example uses the simplest case: unconditional generation of a 100-residue protein with no conditioning information, illustrating the full input-config-output cycle.

In [ ]:
from proto_tools import (
    RFdiffusion3Config,
    RFdiffusion3DesignSpec,
    RFdiffusion3Input,
    run_rfdiffusion3,
)

In [ ]:
# Display input docs
display_api_reference("rfdiffusion3", "input", "run_rfdiffusion3")

# Unconditional design: generate a 100-residue protein from scratch
inputs = RFdiffusion3Input(
    design_specs=[
        RFdiffusion3DesignSpec(length="100")
    ]
)

In [ ]:
# Display config docs
display_api_reference("rfdiffusion3", "config", "run_rfdiffusion3")

# Use a single sample with reduced timesteps to keep the example fast
config = RFdiffusion3Config(
    n_batches=1,
    diffusion_batch_size=1,
    num_timesteps=50,
    step_scale=1.5,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the design
output = run_rfdiffusion3(inputs, config)
n_designs = sum(len(bundle) for bundle in output.designed_structures)
print(f"Generated {n_designs} designs across {len(output)} input specs")

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# Display output docs and inspect results
display_api_reference("rfdiffusion3", "output", "run_rfdiffusion3")

# Output is grouped per input spec: each bundle holds the N designs for one spec.
bundle = output[0]
first = bundle[0]
print(f"Spec key: {bundle.spec_key}")
print(f"Designs in bundle: {len(bundle)}")
print(f"Chains: {len(first.complex.chains)}")
print(f"Sequence length: {len(first.complex.chains[0].sequence)}")
print(f"Sequence: {first.complex.chains[0].sequence}")
print(f"Metadata keys: {list(first.metadata.keys())}")

# Visualize the first designed structure of the first bundle
first.structure.visualize()

## Export Results

Designed structures can be exported to PDB or mmCIF format. PDB is widely compatible with visualization tools such as PyMOL and ChimeraX, while mmCIF supports richer metadata and is preferred for larger or multi-chain structures.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("rfdiffusion3_designs_pdb", export_path=output_dir, file_format="pdb")
print(f"PDB designs exported to {output_dir / 'rfdiffusion3_designs_pdb/'}")

output.export("rfdiffusion3_designs_cif", export_path=output_dir, file_format="cif")
print(f"CIF designs exported to {output_dir / 'rfdiffusion3_designs_cif/'}")